In [ ]:
!git clone https://github.com/GabrielSC92/Transformers_Midterm.git 2>/dev/null; \
    cp Transformers_Midterm/model_2.py . 2>/dev/null || true
%pip install -q torch python-chess datasets huggingface_hub tqdm matplotlib

In [ ]:
import json
import math
import os
import random
import time
from collections import Counter
from pathlib import Path

import chess
import chess.pgn
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from player import (
    BoardTokenizer,
    RecurrentTransformer,
    SQUARE_NAMES,
    MOVE_VOCAB,
    MOVE_TO_IDX,
    NUM_MOVES,
)

device = "cuda"

Device: cuda


## 1. Data Pipeline

We load (FEN, UCI) pairs from `engine_data.jsonl` (Stockfish best-move data from `generate_engine_data.py`) and build a classification dataset.

In [ ]:
ENGINE_DATA_FILE = "engine_data.jsonl"

if not Path(ENGINE_DATA_FILE).exists():
    raise FileNotFoundError(f"Missing {ENGINE_DATA_FILE}. Run generate_engine_data.py first.")

positions = []
with open(ENGINE_DATA_FILE) as f:
    for line in f:
        d = json.loads(line)
        # skip moves we don't have in the vocab (e.g. underpromotions)
        if d["uci"] in MOVE_TO_IDX:
            positions.append((d["fen"], d["uci"]))
print(f"Loaded {len(positions)} (FEN, UCI) pairs from {ENGINE_DATA_FILE}")
print(f"Total positions: {len(positions)}")

Loaded 2000000 (FEN, UCI) pairs from engine_data.jsonl
Total positions: 2000000


In [ ]:
random.seed(42)
random.shuffle(positions)

# 10k val max so eval doesn't take forever, otherwise 5% of data
val_size = min(10_000, len(positions) // 20)
train_positions = positions[val_size:]
val_positions = positions[:val_size]

print(f"Train: {len(train_positions)}, Val: {len(val_positions)}")

Train: 1990000, Val: 10000


## 2. Dataset & DataLoader

In [4]:
tokenizer = BoardTokenizer()

# square name -> index, python-chess order (a1=0 .. h8=63)
SQ_NAME_TO_IDX = {name: i for i, name in enumerate(SQUARE_NAMES)}


class ChessMoveDataset(Dataset):
    """Returns a dict of board tensors + from/to square labels."""

    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        fen, uci = self.data[idx]
        enc = self.tokenizer.encode(fen)
        # e2e4 = from e2, to e4
        from_sq = SQ_NAME_TO_IDX[uci[:2]]
        to_sq = SQ_NAME_TO_IDX[uci[2:4]]

        return {
            "board": torch.tensor(enc["board"], dtype=torch.long),
            "turn": torch.tensor(enc["turn"], dtype=torch.long),
            "castling": torch.tensor(enc["castling"], dtype=torch.long),
            "ep": torch.tensor(enc["ep"], dtype=torch.long),
            "from_sq": from_sq,
            "to_sq": to_sq,
        }


train_ds = ChessMoveDataset(train_positions, tokenizer)
val_ds = ChessMoveDataset(val_positions, tokenizer)

BATCH_SIZE = 512
GRAD_ACCUM_STEPS = 1

# num_workers=0 on Colab/Windows or DataLoader gets weird
NUM_WORKERS = 0 if ("google.colab" in __import__("sys").modules or __import__("sys").platform == "win32") else 4
PIN_MEMORY = torch.cuda.is_available()
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Train batches: 3886, Val batches: 20


## 3. Model

```
FEN → BoardTokenizer → {board(64), turn(1), castling(4), ep(1)}
    → Separate Embeddings + Learned Positional Embeddings
    → Shared Transformer Block × K iterations (with iteration embeddings)
    → Board tokens → From Head (64 scores) + To Head (64 scores)
```

In [5]:
model = RecurrentTransformer(
    d_model=512,
    nhead=8,
    d_ff=2048,
    num_iterations=8,
    dropout=0.1,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1e6:.1f} MB (fp32)")

Total parameters: 3,208,194
Trainable parameters: 3,208,194
Model size: ~12.8 MB (fp32)


## 4. Training

In [6]:
NUM_EPOCHS = 16
EARLY_STOP_PATIENCE = 2
D_MODEL = 512
WARMUP_STEPS = 4000
LABEL_SMOOTHING = 0.1

optimizer = torch.optim.Adam(
    model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9
)

# warmup then inverse-sqrt decay, same idea as the original transformer paper
def vaswani_lr(step):
    step = max(step, 1)
    return D_MODEL ** (-0.5) * min(step ** (-0.5), step * WARMUP_STEPS ** (-1.5))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=vaswani_lr)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

total_steps = NUM_EPOCHS * (len(train_loader) // GRAD_ACCUM_STEPS)
print(f"Total training steps: {total_steps} (optimizer steps, {len(train_loader)} micro-batches/epoch)")

Total training steps: 62176 (optimizer steps, 3886 micro-batches/epoch)


In [9]:
from IPython.display import clear_output
from torch.amp.grad_scaler import GradScaler
from torch.amp.autocast_mode import autocast

history = {"train_loss": [], "val_loss": [], "val_from_acc": [], "val_to_acc": [], "val_move_acc": []}

# bf16 if the GPU supports it, else fp16 - saves memory and speeds things up
scaler = GradScaler("cuda", enabled=(device == "cuda"))
autocast_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

best_val_move = 0.0
epochs_without_improvement = 0


def _batch_to_device(batch, dev):
    return {k: v.to(dev) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    from_correct = to_correct = move_correct = 0
    total = 0
    with torch.no_grad(), autocast(device, dtype=autocast_dtype):
        for batch in loader:
            batch = _batch_to_device(batch, device)
            from_labels = batch.pop("from_sq").to(device)
            to_labels = batch.pop("to_sq").to(device)
            from_logits, to_logits = model(batch)
            loss = criterion(from_logits.float(), from_labels) + criterion(to_logits.float(), to_labels)
            bs = from_labels.size(0)
            total_loss += loss.item() * bs

            from_pred = from_logits.argmax(dim=1)
            to_pred = to_logits.argmax(dim=1)
            from_correct += (from_pred == from_labels).sum().item()
            to_correct += (to_pred == to_labels).sum().item()
            move_correct += ((from_pred == from_labels) & (to_pred == to_labels)).sum().item()
            total += bs

    return total_loss / total, from_correct / total, to_correct / total, move_correct / total


for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    epoch_samples = 0
    t0 = time.time()
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for step_in_epoch, batch in enumerate(pbar):
        batch = _batch_to_device(batch, device)
        from_labels = batch.pop("from_sq").to(device)
        to_labels = batch.pop("to_sq").to(device)

        with autocast(device, dtype=autocast_dtype):
            from_logits, to_logits = model(batch)
            loss = criterion(from_logits.float(), from_labels) + criterion(to_logits.float(), to_labels)
            loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        # actually update weights every GRAD_ACCUM_STEPS; clip grads so it doesn't explode
        if (step_in_epoch + 1) % GRAD_ACCUM_STEPS == 0 or (step_in_epoch + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        bs = from_labels.size(0)
        epoch_loss += loss.item() * GRAD_ACCUM_STEPS * bs
        epoch_samples += bs
        pbar.set_postfix(loss=f"{loss.item()*GRAD_ACCUM_STEPS:.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")

    train_loss = epoch_loss / epoch_samples
    val_loss, val_from, val_to, val_move = evaluate(model, val_loader)
    elapsed = time.time() - t0

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_from_acc"].append(val_from)
    history["val_to_acc"].append(val_to)
    history["val_move_acc"].append(val_move)

    epochs_x = list(range(1, len(history["train_loss"]) + 1))
    clear_output(wait=True)
    for i in range(len(epochs_x)):
        suffix = f" ({elapsed:.0f}s)" if i == len(epochs_x) - 1 else ""
        print(
            f"Epoch {epochs_x[i]}: train={history['train_loss'][i]:.4f} "
            f"val={history['val_loss'][i]:.4f} "
            f"from={history['val_from_acc'][i]:.4f} "
            f"to={history['val_to_acc'][i]:.4f} "
            f"move={history['val_move_acc'][i]:.4f}{suffix}"
        )

    if val_move > best_val_move:
        best_val_move = val_move
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "best_model.pt")
        print(f"  New best model saved (move acc: {val_move:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"  No improvement for {epochs_without_improvement}/{EARLY_STOP_PATIENCE} epochs")
        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print(f"  Early stopping triggered. Best move acc: {best_val_move:.4f}")
            break

Epoch 1: train=3.9863 val=3.8665 from=0.5659 to=0.4599 move=0.3884
Epoch 2: train=3.8878 val=3.8103 from=0.5745 to=0.4685 move=0.4040
Epoch 3: train=3.8181 val=3.7711 from=0.5807 to=0.4786 move=0.4150
Epoch 4: train=3.7652 val=3.7365 from=0.5849 to=0.4850 move=0.4217
Epoch 5: train=3.7205 val=3.7085 from=0.5905 to=0.4876 move=0.4266
Epoch 6: train=3.6836 val=3.6947 from=0.5953 to=0.4943 move=0.4351
Epoch 7: train=3.6514 val=3.6733 from=0.5983 to=0.4995 move=0.4400
Epoch 8: train=3.6231 val=3.6738 from=0.5977 to=0.4960 move=0.4372 (1423s)


  No improvement for 1/2 epochs
  Checkpoint saved (epoch 8)


Epoch 9/16:   0%|          | 0/3886 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 5. Push to HuggingFace

In [ ]:
from huggingface_hub import HfApi, login

login()

REPO_ID = "Izzent/recurrent-transformer-chess"

# use the best model we saved (highest val move acc) for the upload
best_state = torch.load("best_model.pt", map_location=device, weights_only=True)
model.load_state_dict(best_state)
model.eval()
print("Loaded best_model.pt for upload")

config = model.get_config()
with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

with open("tokenizer.json", "w") as f:
    json.dump(tokenizer.to_dict(), f, indent=2)

torch.save(model.state_dict(), "model.pt")

api = HfApi()
api.create_repo(REPO_ID, exist_ok=True)

for fname in ["config.json", "tokenizer.json", "model.pt"]:
    api.upload_file(
        path_or_fileobj=fname,
        path_in_repo=fname,
        repo_id=REPO_ID,
    )
    print(f"Uploaded {fname}")

print(f"\nModel uploaded to https://huggingface.co/{REPO_ID}")

Loaded best_model.pt for upload
Uploaded config.json


No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded tokenizer.json


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded model.pt

Model uploaded to https://huggingface.co/Izzent/recurrent-transformer-chess
